# Préparer les modèles pour le navigateur

L'application doit fonctionner sans réseau, donc tout doit tenir dans le
navigateur. Le classifieur actuel est un DeBERTa-v3-large de 435 millions de
paramètres : plus de 400 Mo même quantifié, impossible à charger sur un terminal
de terrain.

Il faut donc passer à un modèle plus petit. La question est de savoir ce que ça
coûte, et personne ne peut y répondre sans le mesurer. Ce notebook entraîne
plusieurs candidats, compare leur F1 et leur taille une fois quantifiés, et
exporte ce qu'on retient.

Le sélecteur de phrases reste SciBERT, qui fait déjà la bonne taille.

Ce qu'on obtient à la fin, prêt à déposer sur un hébergeur statique : les deux
modèles en ONNX quantifié, leurs tokenizers, et le corpus.

In [ ]:
!pip -q install transformers torch onnx onnxruntime nltk sentencepiece 2>&1 | tail -2
import gc, json, os, random, re, subprocess, tarfile, time, urllib.request, shutil
from collections import Counter
import numpy as np
import torch

random.seed(0); np.random.seed(0); torch.manual_seed(0)
print(torch.cuda.get_device_name(0))

## Les données, comme dans le notebook de vérification

In [ ]:
if not os.path.exists("data/claims_dev.jsonl"):
    urllib.request.urlretrieve(
        "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz", "scifact.tar.gz")
    tarfile.open("scifact.tar.gz").extractall(".")

os.makedirs("evaluate/lib", exist_ok=True)
base = "https://raw.githubusercontent.com/allenai/scifact/master/verisci/evaluate"
urllib.request.urlretrieve(f"{base}/pipeline.py", "evaluate/pipeline.py")
for fichier in ["__init__.py", "data.py", "metrics.py"]:
    urllib.request.urlretrieve(f"{base}/lib/{fichier}", f"evaluate/lib/{fichier}")

articles = {str(json.loads(l)["doc_id"]): json.loads(l)
            for l in open("data/corpus.jsonl", encoding="utf-8")}
tout = [json.loads(l) for l in open("data/claims_train.jsonl", encoding="utf-8")]
developpement = [json.loads(l) for l in open("data/claims_dev.jsonl", encoding="utf-8")]

random.Random(0).shuffle(tout)
coupure = int(0.15 * len(tout))
reglage, entrainement = tout[:coupure], tout[coupure:]
with open("data/claims_reglage.jsonl", "w") as f:
    for a in reglage:
        f.write(json.dumps(a) + "\n")

print(len(articles), "articles,", len(entrainement), "affirmations d'entraînement")

## Recherche et entraînement, repris tels quels

In [ ]:
from nltk.stem.porter import PorterStemmer
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

mots_vides = set('''a an and are as at be but by for if in into is it no not of on
or such that the their then there these they this to was will with'''.split())
racine, deja_vu = PorterStemmer(), {}

def decouper(texte):
    mots = []
    for mot in re.findall(r"[a-z0-9]+", texte.lower()):
        if mot in mots_vides:
            continue
        if mot not in deja_vu:
            deja_vu[mot] = racine.stem(mot)
        mots.append(deja_vu[mot])
    return mots


class BM25:
    def __init__(self, textes, k1=0.9, b=0.4):
        self.k1, self.b = k1, b
        decoupes = [decouper(t) for t in textes]
        self.nb_documents = len(decoupes)
        self.longueurs = np.array([len(d) for d in decoupes], dtype=np.float32)
        self.longueur_moyenne = self.longueurs.mean()
        occurrences = {}
        for i, mots in enumerate(decoupes):
            for mot, frequence in Counter(mots).items():
                occurrences.setdefault(mot, []).append((i, frequence))
        self.index = {}
        for mot, liste in occurrences.items():
            ou = np.array([x[0] for x in liste], dtype=np.int32)
            combien = np.array([x[1] for x in liste], dtype=np.float32)
            df = len(liste)
            self.index[mot] = (ou, combien,
                               np.log(1 + (self.nb_documents - df + 0.5) / (df + 0.5)))

    def noter(self, question):
        notes = np.zeros(self.nb_documents, dtype=np.float32)
        for mot in decouper(question):
            if mot not in self.index:
                continue
            ou, combien, rarete = self.index[mot]
            longueur = 1 - self.b + self.b * self.longueurs[ou] / self.longueur_moyenne
            notes[ou] += rarete * combien * (self.k1 + 1) / (combien + self.k1 * longueur)
        return notes


identifiants = list(articles)
bm25 = BM25([f"{articles[d]['title']} {' '.join(articles[d]['abstract'])}" for d in identifiants])

def retrouver(affirmations, combien=10):
    trouve = {}
    for a in affirmations:
        notes = bm25.noter(a["claim"])
        haut = np.argpartition(-notes, combien)[:combien]
        trouve[a["id"]] = [identifiants[i] for i in haut[np.argsort(-notes[haut])]]
    return trouve

retrouves = {"reglage": retrouver(reglage), "developpement": retrouver(developpement)}


def charger_modele(nom, nb_classes=None):
    options = {"num_labels": nb_classes} if nb_classes else {}
    return AutoModelForSequenceClassification.from_pretrained(nom, **options).float().to("cuda")


class Exemples(Dataset):
    def __init__(self, e, c): self.e, self.c = e, c
    def __len__(self): return len(self.e)
    def __getitem__(self, i): return self.e[i], int(self.c[i])


def entrainer(modele, tokenizer, entrees, cibles, epoques=3, taille_lot=32,
              pas=2e-5, longueur=256, poids_classes=None):
    def assembler(lot):
        paires = [x[0] for x in lot]
        y = torch.tensor([x[1] for x in lot])
        return tokenizer([a for a, _ in paires], [b for _, b in paires], padding=True,
                         truncation=True, max_length=longueur, return_tensors="pt"), y

    donnees = DataLoader(Exemples(entrees, cibles), batch_size=taille_lot,
                         shuffle=True, collate_fn=assembler)
    optimiseur = torch.optim.AdamW(modele.parameters(), lr=pas, weight_decay=0.01)
    etapes = len(donnees) * epoques
    calendrier = get_linear_schedule_with_warmup(optimiseur, int(0.1 * etapes), etapes)
    critere = torch.nn.CrossEntropyLoss(
        weight=poids_classes.to("cuda") if poids_classes is not None else None)
    echelle = torch.amp.GradScaler("cuda")
    modele.train()
    for epoque in range(epoques):
        cumul = 0.0
        for encode, cible in donnees:
            encode = {k: v.to("cuda") for k, v in encode.items()}
            cible = cible.to("cuda")
            optimiseur.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                perte = critere(modele(**encode).logits, cible)
            echelle.scale(perte).backward()
            echelle.unscale_(optimiseur)
            torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
            echelle.step(optimiseur); echelle.update(); calendrier.step()
            cumul += perte.item()
        print(f"  époque {epoque+1} : perte {cumul/len(donnees):.4f}")
    modele.eval()
    return modele


@torch.no_grad()
def predire(modele, tokenizer, paires, taille_lot=128, longueur=256, colonne=1):
    sorties = []
    for i in range(0, len(paires), taille_lot):
        lot = paires[i:i + taille_lot]
        encode = tokenizer([a for a, _ in lot], [b for _, b in lot], padding=True,
                           truncation=True, max_length=longueur,
                           return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            proba = torch.softmax(modele(**encode).logits.float(), dim=-1)
        sorties.append(proba[:, colonne].cpu().numpy() if colonne is not None
                       else proba.cpu().numpy())
    if not sorties:
        return np.zeros(0)
    return np.concatenate(sorties) if colonne is not None else np.vstack(sorties)

## Le sélecteur de phrases, inchangé

In [ ]:
def phrases_et_etiquettes(affirmations):
    entrees, cibles = [], []
    for a in affirmations:
        preuves = a.get("evidence") or {}
        for article in a.get("cited_doc_ids", []):
            article = str(article)
            if article not in articles:
                continue
            bonnes = {i for g in preuves.get(article, []) for i in g["sentences"]}
            for i, phrase in enumerate(articles[article]["abstract"]):
                entrees.append((phrase, a["claim"]))
                cibles.append(1 if i in bonnes else 0)
    return entrees, np.array(cibles)


phrases_ent, vraies_ent = phrases_et_etiquettes(entrainement)
phrases_reg, vraies_reg = phrases_et_etiquettes(reglage)

poids = torch.tensor([1.0, float((vraies_ent == 0).sum() / max((vraies_ent == 1).sum(), 1))])
tokenizer_selecteur = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
selecteur = charger_modele("allenai/scibert_scivocab_uncased", nb_classes=2)
selecteur = entrainer(selecteur, tokenizer_selecteur, phrases_ent, vraies_ent,
                      poids_classes=poids)

def f1_binaire(notes, vraies, seuil):
    r = notes >= seuil
    j = int((r & (vraies == 1)).sum()); f = int((r & (vraies == 0)).sum())
    m = int((~r & (vraies == 1)).sum())
    p = j / (j + f) if j + f else 0.0
    rr = j / (j + m) if j + m else 0.0
    return 2 * p * rr / (p + rr) if p + rr else 0.0

notes_reg = predire(selecteur, tokenizer_selecteur, phrases_reg)
seuil_phrases = float(max(np.arange(0.05, 0.96, 0.05),
                          key=lambda s: f1_binaire(notes_reg, vraies_reg, s)))
print(f"seuil retenu : {seuil_phrases:.2f}")

## Les candidats pour le classifieur

DeBERTa-v3-base a un vocabulaire de 128 000 entrées, ce qui gonfle beaucoup sa
taille une fois quantifié. DistilRoBERTa en a 50 000 et pèse bien moins. On
mesure les deux : F1 de la chaîne complète, et taille réelle du fichier ONNX.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

def exporter(modele, tokenizer, dossier):
    os.makedirs(dossier, exist_ok=True)
    modele = modele.cpu().eval()
    exemple = tokenizer(["phrase de test"], ["affirmation de test"],
                        return_tensors="pt", padding=True)
    entrees = ["input_ids", "attention_mask"]
    args = (exemple["input_ids"], exemple["attention_mask"])
    if "token_type_ids" in exemple:
        entrees.append("token_type_ids")
        args = args + (exemple["token_type_ids"],)

    axes = {n: {0: "lot", 1: "longueur"} for n in entrees}
    axes["logits"] = {0: "lot"}
    torch.onnx.export(modele, args, f"{dossier}/model.onnx",
                      input_names=entrees, output_names=["logits"],
                      dynamic_axes=axes, opset_version=14,
                      do_constant_folding=True, dynamo=False)
    quantize_dynamic(f"{dossier}/model.onnx", f"{dossier}/model_quantise.onnx",
                     weight_type=QuantType.QUInt8, per_channel=True, reduce_range=True)
    os.remove(f"{dossier}/model.onnx")
    tokenizer.save_pretrained(dossier)
    modele.config.save_pretrained(dossier)
    modele.to("cuda")
    return os.path.getsize(f"{dossier}/model_quantise.onnx") / 1e6


taille_selecteur = exporter(selecteur, tokenizer_selecteur, "export/selecteur")
print(f"sélecteur SciBERT : {taille_selecteur:.0f} Mo\n")

nb_phrases_max = 3

def choisir_phrases(affirmation, article, seuil):
    phrases = articles[article]["abstract"]
    if not phrases:
        return []
    notes = predire(selecteur, tokenizer_selecteur, [(p, affirmation) for p in phrases])
    retenues = np.where(notes >= seuil)[0]
    if len(retenues) == 0:
        retenues = np.array([int(notes.argmax())])
    retenues = retenues[np.argsort(-notes[retenues])][:nb_phrases_max]
    return sorted(int(i) for i in retenues)


def exemples_etiquette(affirmations, seuil):
    entrees, etiquettes = [], []
    for a in affirmations:
        preuves = a.get("evidence") or {}
        for article in a.get("cited_doc_ids", []):
            article = str(article)
            if article not in articles:
                continue
            indices = choisir_phrases(a["claim"], article, seuil)
            if not indices:
                continue
            entrees.append((" ".join(articles[article]["abstract"][i] for i in indices),
                            a["claim"]))
            g = preuves.get(article)
            etiquettes.append(g[0]["label"] if g else "NOINFO")
    return entrees, etiquettes


entrees_ent, etiquettes_ent = exemples_etiquette(entrainement, seuil_phrases)
print(len(entrees_ent), "exemples :", dict(Counter(etiquettes_ent)))


def evaluer(resultats, fichier_or):
    with open("predictions.jsonl", "w") as f:
        for ligne in resultats:
            f.write(json.dumps(ligne) + "\n")
    sortie = subprocess.run(
        ["python", "pipeline.py", "--gold", f"../{fichier_or}", "--corpus",
         "../data/corpus.jsonl", "--prediction", "../predictions.jsonl",
         "--output", "../mesures.json"], cwd="evaluate", capture_output=True, text=True)
    if not os.path.exists("mesures.json"):
        print(sortie.stdout[-800:], sortie.stderr[-800:])
        raise RuntimeError("évaluateur en échec")
    m = json.load(open("mesures.json")); os.remove("mesures.json")
    return m


def verifier(affirmations, trouves, classifieur, tokenizer, vers_etiquette,
             seuil, nb_articles):
    resultats = []
    for a in affirmations:
        preuve = {}
        for article in trouves[a["id"]][:nb_articles]:
            indices = choisir_phrases(a["claim"], article, seuil)
            if not indices:
                continue
            texte = " ".join(articles[article]["abstract"][i] for i in indices)
            proba = predire(classifieur, tokenizer, [(texte, a["claim"])],
                            taille_lot=1, longueur=320, colonne=None)[0]
            etiquette = vers_etiquette[int(proba.argmax())]
            if etiquette != "NOINFO":
                preuve[str(article)] = {"label": etiquette, "sentences": indices}
        resultats.append({"id": a["id"], "evidence": preuve})
    return resultats


# Les trois paliers proposes dans l'application. Le rapide est telecharge par
# defaut, les deux autres a la demande.
candidats = {
    "rapide":        "cross-encoder/nli-distilroberta-base",
    "intermediaire": "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    "precis":        "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
}

mesures_candidats = {}
for libelle, nom in candidats.items():
    print(f"\n=== {libelle} : {nom} ===")
    tk = AutoTokenizer.from_pretrained(nom)
    md_ = charger_modele(nom)
    sorties = {int(k): v.lower() for k, v in md_.config.id2label.items()}
    vers_sortie = {
        "SUPPORT":    next(i for i, v in sorties.items() if v.startswith("entail")),
        "NOINFO":     next(i for i, v in sorties.items() if v.startswith("neutral")),
        "CONTRADICT": next(i for i, v in sorties.items() if v.startswith("contradic")),
    }
    vers_etiquette = {v: k for k, v in vers_sortie.items()}
    cibles = np.array([vers_sortie[e] for e in etiquettes_ent])
    md_ = entrainer(md_, tk, entrees_ent, cibles, taille_lot=4, pas=1e-5, longueur=320)

    m = evaluer(verifier(developpement, retrouves["developpement"], md_, tk,
                         vers_etiquette, seuil_phrases, 3), "data/claims_dev.jsonl")
    f1 = m["abstract_rationalized"]["f1"] * 100
    parametres = sum(p.numel() for p in md_.parameters()) / 1e6

    # Exporter tout de suite : les trois modeles ensemble ne tiendraient pas
    # en memoire, le precis pesant a lui seul 435 millions de parametres.
    taille = exporter(md_, tk, f"export/classifieur_{libelle}")

    mesures_candidats[libelle] = {"nom": nom, "f1": f1, "parametres": parametres,
                                  "taille": taille}
    print(f"  F1 {f1:.1f}   {parametres:.0f} M paramètres   {taille:.0f} Mo en ONNX")

    del md_
    gc.collect()
    torch.cuda.empty_cache()

print()
print(f"{'palier':<16}{'F1':>7}{'paramètres':>13}{'ONNX':>10}")
for libelle, d in mesures_candidats.items():
    print(f"{libelle:<16}{d['f1']:>7.1f}{d['parametres']:>11.0f} M{d['taille']:>7.0f} Mo")

## Le corpus pour le navigateur

L'index BM25 se reconstruit en quelques secondes côté navigateur, donc on
n'envoie que le corpus. Les phrases sont déjà découpées dans le jeu de données.

In [ ]:
corpus_navigateur = [
    {"id": identifiant,
     "titre": articles[identifiant]["title"],
     "phrases": articles[identifiant]["abstract"]}
    for identifiant in identifiants
]
with open("export/corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus_navigateur, f, ensure_ascii=False, separators=(",", ":"))

taille_corpus = os.path.getsize("export/corpus.json") / 1e6
print(f"corpus : {len(corpus_navigateur)} articles, {taille_corpus:.1f} Mo")

reglages = {
    "seuil_phrases": seuil_phrases,
    "nb_phrases_max": nb_phrases_max,
    "nb_articles": 3,
    "bm25": {"k1": 0.9, "b": 0.4},
    "mots_vides": sorted(mots_vides),
}
json.dump(reglages, open("export/reglages.json", "w"), indent=2)

## Le paquet à déposer sur l'hébergeur

In [ ]:
resume = {libelle: {"nom": d["nom"], "f1": d["f1"], "parametres_M": d["parametres"],
                    "taille_Mo": d.get("taille")}
          for libelle, d in mesures_candidats.items()}
json.dump(resume, open("export/mesures_modeles.json", "w"), indent=2)

!zip -qr export.zip export
print(f"export.zip : {os.path.getsize('export.zip')/1e6:.0f} Mo\n")
for chemin, _, fichiers in os.walk("export"):
    for f in sorted(fichiers):
        entier = os.path.join(chemin, f)
        print(f"  {entier:<46}{os.path.getsize(entier)/1e6:>8.1f} Mo")

from google.colab import files
files.download("export.zip")